In [ ]:
import pandas as pd
import numpy as np
import pubchempy as pcp
from functools import lru_cache
import time
from rdkit import Chem

In [ ]:
df = pd.read_csv("Cleaned_VLE_Data.csv")

In [ ]:
print(df.shape)
df.head()

(110383, 8)


,property,value,phase,"Temperature, K","Pressure, kPa",Mole fraction,Component 1,Component 2
0,"Thermal conductivity, W/m/K",0.02096,Gas,300.0,724.0,0.2493,carbon dioxide,methane
1,"Thermal conductivity, W/m/K",0.02125,Gas,300.0,1288.0,0.2493,carbon dioxide,methane
2,"Thermal conductivity, W/m/K",0.02161,Gas,300.0,1835.0,0.2493,carbon dioxide,methane
3,"Thermal conductivity, W/m/K",0.02197,Gas,300.0,2335.0,0.2493,carbon dioxide,methane
4,"Thermal conductivity, W/m/K",0.02245,Gas,300.0,2820.0,0.2493,carbon dioxide,methane


In [ ]:
@lru_cache(maxsize=None)
def get_smiles(name, retries=5):
    for i in range(retries):
        try:
            compounds = pcp.get_compounds(name, "name")
            if compounds:
                return compounds[0].smiles
            return None
        except Exception as e:
            if "SSL" in str(e) or "EOF" in str(e):
                print(f"SSL Error for {name}, retrying ({i+1}/{retries})...")
                time.sleep(2)  # Give the connection a moment to breathe
                continue
            print(f"Permanent Error fetching {name}: {e}")
            break
    return None


df["Smiles 1"] = df["Component 1"].apply(get_smiles)
df["Smiles 2"] = df["Component 2"].apply(get_smiles)

In [ ]:
print(df.shape)
df.head()

(110383, 10)


,property,value,phase,"Temperature, K","Pressure, kPa",Mole fraction,Component 1,Component 2,Smiles 1,Smiles 2
0,"Thermal conductivity, W/m/K",0.02096,Gas,300.0,724.0,0.2493,carbon dioxide,methane,C(=O)=O,C
1,"Thermal conductivity, W/m/K",0.02125,Gas,300.0,1288.0,0.2493,carbon dioxide,methane,C(=O)=O,C
2,"Thermal conductivity, W/m/K",0.02161,Gas,300.0,1835.0,0.2493,carbon dioxide,methane,C(=O)=O,C
3,"Thermal conductivity, W/m/K",0.02197,Gas,300.0,2335.0,0.2493,carbon dioxide,methane,C(=O)=O,C
4,"Thermal conductivity, W/m/K",0.02245,Gas,300.0,2820.0,0.2493,carbon dioxide,methane,C(=O)=O,C


In [ ]:
# Replace empty strings with NaN across the specific columns
df["Smiles 1"] = df["Smiles 1"].replace("", np.nan)
df["Smiles 2"] = df["Smiles 2"].replace("", np.nan)

# Now the original, simpler code works for all cases
empty_count = (df["Smiles 1"].isnull() | df["Smiles 2"].isnull()).sum()

print(f"Number of rows with empty/NaN 'Smiles 1' or 'Smiles 2': {empty_count}")

Number of rows with empty/NaN 'Smiles 1' or 'Smiles 2': 2241


In [ ]:
df = df.dropna(subset=["Smiles 1", "Smiles 2"])
print(df.shape)
df.head()

(108142, 10)


,property,value,phase,"Temperature, K","Pressure, kPa",Mole fraction,Component 1,Component 2,Smiles 1,Smiles 2
0,"Thermal conductivity, W/m/K",0.02096,Gas,300.0,724.0,0.2493,carbon dioxide,methane,C(=O)=O,C
1,"Thermal conductivity, W/m/K",0.02125,Gas,300.0,1288.0,0.2493,carbon dioxide,methane,C(=O)=O,C
2,"Thermal conductivity, W/m/K",0.02161,Gas,300.0,1835.0,0.2493,carbon dioxide,methane,C(=O)=O,C
3,"Thermal conductivity, W/m/K",0.02197,Gas,300.0,2335.0,0.2493,carbon dioxide,methane,C(=O)=O,C
4,"Thermal conductivity, W/m/K",0.02245,Gas,300.0,2820.0,0.2493,carbon dioxide,methane,C(=O)=O,C


In [ ]:
df["mol1"] = df["Smiles 1"].apply(Chem.MolFromSmiles)
df["mol2"] = df["Smiles 2"].apply(Chem.MolFromSmiles)

[14:41:59] WARNING: not removing hydrogen atom without neighbors
[14:41:59] WARNING: not removing hydrogen atom without neighbors
[14:41:59] WARNING: not removing hydrogen atom without neighbors
[14:41:59] WARNING: not removing hydrogen atom without neighbors
[14:41:59] WARNING: not removing hydrogen atom without neighbors
[14:41:59] WARNING: not removing hydrogen atom without neighbors
[14:41:59] WARNING: not removing hydrogen atom without neighbors
[14:41:59] WARNING: not removing hydrogen atom without neighbors
[14:41:59] WARNING: not removing hydrogen atom without neighbors
[14:41:59] WARNING: not removing hydrogen atom without neighbors
[14:41:59] WARNING: not removing hydrogen atom without neighbors
[14:41:59] WARNING: not removing hydrogen atom without neighbors
[14:41:59] WARNING: not removing hydrogen atom without neighbors
[14:41:59] WARNING: not removing hydrogen atom without neighbors
[14:41:59] WARNING: not removing hydrogen atom without neighbors
[14:41:59] WARNING: not r

In [ ]:
# Replace empty strings with NaN across the specific columns
df["mol1"] = df["mol1"].replace("", np.nan)
df["mol2"] = df["mol2"].replace("", np.nan)

# Now the original, simpler code works for all cases
empty_count = (df["mol1"].isnull() | df["mol2"].isnull()).sum()

print(f"Number of rows with empty/NaN 'mol1' or 'mol2': {empty_count}")

Number of rows with empty/NaN 'mol1' or 'mol2': 0


In [ ]:
print(df.shape)
df.head()
df.to_csv("Cleaned_VLE_Data_with_Smiles_and_Mols.csv", index=False)

(108142, 12)
